In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# モデルとトークナイザーの読み込み
model_id = "llm-jp/llm-jp-3-150m-instruct3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# プロンプトの設定
prompt = "The movie was full of"

# 入力のトークン化
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

# 生成パラメータの設定
gen_kwargs = {
    "max_new_tokens": 10,  # 短めに設定
    "do_sample": True,
    "temperature": 1.0,
    "return_dict_in_generate": True,
    "output_scores": True,
}

# テキスト生成
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids, attention_mask=attention_mask, **gen_kwargs
    )

# 生成されたトークンとその尤度を取得
generated_ids = outputs.sequences[0]
scores = outputs.scores

# 結果の表示
print("生成されたテキストと各単語の尤度:")
current_text = prompt
for i, (token_id, score) in enumerate(zip(generated_ids[len(input_ids[0]) :], scores)):
    token = tokenizer.decode([token_id])
    # 尤度の計算（softmaxを適用）
    probabilities = torch.softmax(score, dim=-1)
    token_prob = probabilities[0, token_id].item()

    # トークンの間に半角スペースを追加
    if not token.startswith(" ") and current_text[-1] != " ":
        current_text += " "
    current_text += token

    print(f"{current_text}")
    print(f"単語: {token}, 尤度: {token_prob:.4f}")
    print("-" * 50)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


生成されたテキストと各単語の尤度:
The movie was full of memorable
単語: memorable, 尤度: 0.1116
--------------------------------------------------
The movie was full of memorable moments
単語: moments, 尤度: 0.7249
--------------------------------------------------
The movie was full of memorable moments and
単語: and, 尤度: 0.3406
--------------------------------------------------
The movie was full of memorable moments and memorable
単語: memorable, 尤度: 0.2725
--------------------------------------------------
The movie was full of memorable moments and memorable characters
単語: characters, 尤度: 0.5208
--------------------------------------------------
The movie was full of memorable moments and memorable characters .
単語: ., 尤度: 0.3620
--------------------------------------------------
The movie was full of memorable moments and memorable characters . We
単語: We, 尤度: 0.0065
--------------------------------------------------
The movie was full of memorable moments and memorable characters . We couldn
単語: couldn, 尤度: 